In [2]:
# ============================================================================
# MARKETLAB DAY 3 - SHAP ANALYSIS & EXPLAINABILITY
# ============================================================================

print("🔥 MARKETLAB DAY 3 - SHAP ANALYSIS & EXPLAINABILITY")
print("="*80)

# Mount Google Drive
print("\n📁 Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted successfully!")

# Install SHAP
print("\n📦 Installing SHAP library...")
!pip install shap -q
print("✅ SHAP installed!")

# Import libraries
print("\n📚 Importing libraries...")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# TensorFlow and sklearn
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
import pickle

print("✅ All libraries imported!")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n" + "="*80)
print("✅ SETUP COMPLETE!")
print("✅ Ready to run CELL 2!")
print("="*80)

🔥 MARKETLAB DAY 3 - SHAP ANALYSIS & EXPLAINABILITY

📁 Mounting Google Drive...
Mounted at /content/drive
✅ Drive mounted successfully!

📦 Installing SHAP library...
✅ SHAP installed!

📚 Importing libraries...
✅ All libraries imported!

✅ SETUP COMPLETE!
✅ Ready to run CELL 2!


In [2]:
# ============================================================================
# MARKETLAB DAY 3 - SHAP ANALYSIS (FIXED - ALL IMPORTS INCLUDED)
# ============================================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# CRITICAL: Import RandomForestRegressor!
from sklearn.ensemble import RandomForestRegressor

print("🔥 MARKETLAB DAY 3 - SHAP EXPLAINABILITY ANALYSIS")
print("="*80)
print("📊 Analyzing feature importance across 9 stocks")
print("🎯 Understanding which features drive predictions")
print("="*80 + "\n")

# ============================================================================
# CONFIGURATION
# ============================================================================

# 9 stocks (TATAMOTORS replaced with MARUTI)
STOCKS = [
    'SUNPHARMA.NS',
    'ITC.NS',
    'TCS.NS',
    'RELIANCE.NS',
    'INFY.NS',
    'COALINDIA.NS',
    'BAJFINANCE.NS',
    'HDFCBANK.NS',
    'MARUTI.NS'
]

# Paths
BASE_PATH = Path('/content/drive/MyDrive/MarketLab_BEAST')
DATA_PATH = BASE_PATH / 'stock_data'
RESULTS_DAY1 = BASE_PATH / 'results_day1'
RESULTS_DAY2 = BASE_PATH / 'results_day2'
RESULTS_DAY3 = BASE_PATH / 'results_day3'
RESULTS_DAY3.mkdir(exist_ok=True, parents=True)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_csv(ticker):
    """Load stock data"""
    path = DATA_PATH / f"{ticker}.csv"

    with open(path, 'r') as f:
        first_line = f.readline()

    if 'Price","Open' in first_line:
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
        df = df.set_index('Date')
        df = df.rename(columns={'Price': 'Close', 'Vol.': 'Volume'})
        if df['Volume'].dtype == 'object':
            df['Volume'] = df['Volume'].str.replace('M', '').str.replace('K', '').str.replace(',', '')
            df['Volume'] = df['Volume'].apply(lambda x: float(x)*1000000 if 'M' in str(x) else (float(x)*1000 if 'K' in str(x) else float(x) if x else 0))
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    else:
        df = pd.read_csv(path, skiprows=[1])
        df = df.iloc[1:]
        df['Date'] = pd.to_datetime(df['Price'])
        df = df.set_index('Date')
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df.dropna().sort_index()

def generate_325_features(df):
    """Generate all 325 features (same as Day 2)"""

    # Price features
    df['hl_ratio'] = df['High'] / df['Low']
    df['co_ratio'] = df['Close'] / df['Open']
    df['oc_diff'] = df['Open'] - df['Close']
    df['hl_diff'] = df['High'] - df['Low']
    df['body_size'] = abs(df['Close'] - df['Open'])
    df['upper_shadow'] = df['High'] - df[['Open','Close']].max(axis=1)
    df['lower_shadow'] = df[['Open','Close']].min(axis=1) - df['Low']
    df['body_position'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])

    # Returns
    for p in [1,2,3,5,7,10,14,21,30,60,90,120,180,252]:
        df[f'ret_{p}'] = df['Close'].pct_change(p)
        df[f'log_ret_{p}'] = np.log(df['Close'] / df['Close'].shift(p))

    # Moving averages
    for p in [3,5,7,10,14,20,30,50,100,150,200]:
        df[f'sma_{p}'] = df['Close'].rolling(p).mean()
        df[f'ema_{p}'] = df['Close'].ewm(span=p).mean()
        df[f'close_sma_ratio_{p}'] = df['Close'] / df[f'sma_{p}']
        df[f'close_ema_ratio_{p}'] = df['Close'] / df[f'ema_{p}']

    # Volatility
    for p in [5,10,20,30,60,90]:
        df[f'volatility_{p}'] = df['Close'].pct_change().rolling(p).std()
        df[f'volatility_{p}_annual'] = df[f'volatility_{p}'] * np.sqrt(252)
        df[f'parkinson_{p}'] = np.sqrt((1/(4*np.log(2))) * ((np.log(df['High']/df['Low']))**2).rolling(p).mean())
        df[f'garman_klass_{p}'] = np.sqrt(0.5 * ((np.log(df['High']/df['Low']))**2).rolling(p).mean() - (2*np.log(2)-1) * ((np.log(df['Close']/df['Open']))**2).rolling(p).mean())

    # RSI
    for p in [7,9,14,21,28]:
        delta = df['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(p).mean()
        loss = -delta.where(delta < 0, 0).rolling(p).mean()
        rs = gain / loss
        df[f'rsi_{p}'] = 100 - (100 / (1 + rs))
        df[f'rsi_{p}_slope'] = df[f'rsi_{p}'].diff(5)
        df[f'rsi_{p}_ema'] = df[f'rsi_{p}'].ewm(span=5).mean()

    # MACD
    for fast, slow, signal in [(12,26,9), (5,35,5), (19,39,9)]:
        ema_fast = df['Close'].ewm(span=fast).mean()
        ema_slow = df['Close'].ewm(span=slow).mean()
        df[f'macd_{fast}_{slow}'] = ema_fast - ema_slow
        df[f'macd_signal_{fast}_{slow}_{signal}'] = df[f'macd_{fast}_{slow}'].ewm(span=signal).mean()
        df[f'macd_hist_{fast}_{slow}_{signal}'] = df[f'macd_{fast}_{slow}'] - df[f'macd_signal_{fast}_{slow}_{signal}']
        df[f'macd_slope_{fast}_{slow}'] = df[f'macd_{fast}_{slow}'].diff(3)

    # Bollinger Bands
    for p in [10,20,30,50]:
        for std_mult in [1.5, 2.0, 2.5]:
            sma = df['Close'].rolling(p).mean()
            std = df['Close'].rolling(p).std()
            df[f'bb_upper_{p}_{std_mult}'] = sma + (std_mult * std)
            df[f'bb_lower_{p}_{std_mult}'] = sma - (std_mult * std)
            df[f'bb_width_{p}_{std_mult}'] = (df[f'bb_upper_{p}_{std_mult}'] - df[f'bb_lower_{p}_{std_mult}']) / sma
            df[f'bb_position_{p}_{std_mult}'] = (df['Close'] - df[f'bb_lower_{p}_{std_mult}']) / (df[f'bb_upper_{p}_{std_mult}'] - df[f'bb_lower_{p}_{std_mult}'])

    # Volume
    for p in [5,10,20,30,50]:
        df[f'volume_sma_{p}'] = df['Volume'].rolling(p).mean()
        df[f'volume_ratio_{p}'] = df['Volume'] / df[f'volume_sma_{p}']
        df[f'volume_std_{p}'] = df['Volume'].rolling(p).std()
        df[f'volume_cv_{p}'] = df[f'volume_std_{p}'] / df[f'volume_sma_{p}']
        df[f'obv_{p}'] = (np.sign(df['Close'].diff()) * df['Volume']).rolling(p).sum()
        df[f'vwap_{p}'] = (df['Close'] * df['Volume']).rolling(p).sum() / df['Volume'].rolling(p).sum()

    # Momentum
    for p in [5,10,20,30,50]:
        df[f'momentum_{p}'] = df['Close'] - df['Close'].shift(p)
        df[f'roc_{p}'] = df['Close'].pct_change(p) * 100
        df[f'trix_{p}'] = df['Close'].ewm(span=p).mean().ewm(span=p).mean().ewm(span=p).mean().pct_change()

    # ATR
    for p in [7,14,21,30]:
        high_low = df['High'] - df['Low']
        high_close = np.abs(df['High'] - df['Close'].shift())
        low_close = np.abs(df['Low'] - df['Close'].shift())
        ranges = pd.concat([high_low, high_close, low_close], axis=1)
        true_range = ranges.max(axis=1)
        df[f'atr_{p}'] = true_range.rolling(p).mean()
        df[f'atr_ratio_{p}'] = df[f'atr_{p}'] / df['Close']
        df[f'natr_{p}'] = (df[f'atr_{p}'] / df['Close']) * 100

    # Statistical
    for p in [10,20,30,50]:
        df[f'std_{p}'] = df['Close'].rolling(p).std()
        df[f'skew_{p}'] = df['Close'].rolling(p).skew()
        df[f'kurt_{p}'] = df['Close'].rolling(p).kurt()
        df[f'median_{p}'] = df['Close'].rolling(p).median()
        df[f'quantile_25_{p}'] = df['Close'].rolling(p).quantile(0.25)
        df[f'quantile_75_{p}'] = df['Close'].rolling(p).quantile(0.75)
        df[f'iqr_{p}'] = df[f'quantile_75_{p}'] - df[f'quantile_25_{p}']

    # Lags
    for lag in [1,2,3,5,7,10,14,21,30]:
        df[f'close_lag_{lag}'] = df['Close'].shift(lag)
        df[f'volume_lag_{lag}'] = df['Volume'].shift(lag)
        df[f'high_lag_{lag}'] = df['High'].shift(lag)
        df[f'low_lag_{lag}'] = df['Low'].shift(lag)

    # Support/Resistance
    for p in [10,20,50,100]:
        df[f'high_max_{p}'] = df['High'].rolling(p).max()
        df[f'low_min_{p}'] = df['Low'].rolling(p).min()
        df[f'distance_high_{p}'] = (df['Close'] - df[f'high_max_{p}']) / df['Close']
        df[f'distance_low_{p}'] = (df['Close'] - df[f'low_min_{p}']) / df['Close']

    # Stochastic
    for p in [14,21]:
        low_min = df['Low'].rolling(p).min()
        high_max = df['High'].rolling(p).max()
        df[f'stoch_k_{p}'] = 100 * (df['Close'] - low_min) / (high_max - low_min)
        df[f'stoch_d_{p}'] = df[f'stoch_k_{p}'].rolling(3).mean()

    # Ichimoku
    high_9 = df['High'].rolling(9).max()
    low_9 = df['Low'].rolling(9).min()
    df['tenkan_sen'] = (high_9 + low_9) / 2
    high_26 = df['High'].rolling(26).max()
    low_26 = df['Low'].rolling(26).min()
    df['kijun_sen'] = (high_26 + low_26) / 2
    df['senkou_span_a'] = ((df['tenkan_sen'] + df['kijun_sen']) / 2).shift(26)
    high_52 = df['High'].rolling(52).max()
    low_52 = df['Low'].rolling(52).min()
    df['senkou_span_b'] = ((high_52 + low_52) / 2).shift(26)
    df['chikou_span'] = df['Close'].shift(-26)

    # Clean infinity/NaN
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(method='ffill').fillna(method='bfill').fillna(0)

    return df

# ============================================================================
# ANALYZE USING RANDOM FOREST (FASTER THAN DL FOR SHAP)
# ============================================================================

def analyze_stock_features(ticker):
    """Analyze feature importance for one stock using Random Forest"""

    print(f"\n{'='*80}")
    print(f"Stock: {ticker}")
    print(f"{'='*80}")

    try:
        # Load data
        print("   📥 Loading data...")
        df = load_csv(ticker)
        print(f"      ✅ {len(df)} rows loaded")

        # Generate features
        print("   🔧 Generating features...")
        df = generate_325_features(df)
        df = df.dropna()

        # Prepare data
        exclude = ['Open', 'High', 'Low', 'Close', 'Volume']
        feature_cols = [c for c in df.columns if c not in exclude]

        X = df[feature_cols].values
        y = df['Close'].values

        # Train/test split (80/20)
        split_idx = int(0.8 * len(X))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        print(f"      ✅ Train: {len(X_train)} | Test: {len(X_test)}")

        # Train Random Forest (fast, interpretable)
        print("   🌲 Training Random Forest...")
        rf = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        score = rf.score(X_test, y_test)
        print(f"      ✅ RF R² = {score:.4f}")

        # SHAP analysis
        print("   🔍 Computing SHAP values...")
        explainer = shap.TreeExplainer(rf)

        # Sample 100 test samples for SHAP (faster)
        sample_size = min(100, len(X_test))
        sample_idx = np.random.choice(len(X_test), sample_size, replace=False)
        X_sample = X_test[sample_idx]

        shap_values = explainer.shap_values(X_sample)
        print(f"      ✅ SHAP computed for {sample_size} samples")

        # Feature importance from SHAP
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': mean_abs_shap
        }).sort_values('importance', ascending=False)

        # Top 30 features
        top_30 = feature_importance.head(30)

        print("\n   🏆 TOP 10 MOST IMPORTANT FEATURES:")
        for idx, row in top_30.head(10).iterrows():
            print(f"      {row['feature']:30s} {row['importance']:.6f}")

        # Save results
        clean_ticker = ticker.replace('.NS', '')

        # Save feature importance
        feature_importance.to_csv(
            RESULTS_DAY3 / f"{clean_ticker}_feature_importance.csv",
            index=False
        )

        # Create and save SHAP summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(
            shap_values,
            X_sample,
            feature_names=feature_cols,
            max_display=20,
            show=False
        )
        plt.title(f'SHAP Feature Importance - {clean_ticker}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(RESULTS_DAY3 / f"{clean_ticker}_shap_summary.png", dpi=300, bbox_inches='tight')
        plt.close()

        # Create feature importance bar plot
        plt.figure(figsize=(12, 10))
        top_20 = top_30.head(20)
        plt.barh(range(len(top_20)), top_20['importance'].values)
        plt.yticks(range(len(top_20)), top_20['feature'].values)
        plt.xlabel('Mean |SHAP Value|', fontsize=12)
        plt.title(f'Top 20 Features - {clean_ticker}', fontsize=16, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(RESULTS_DAY3 / f"{clean_ticker}_top20_features.png", dpi=300, bbox_inches='tight')
        plt.close()

        print(f"   ✅ DONE! Plots saved.")

        return {
            'ticker': ticker,
            'rf_r2': score,
            'top_30_features': top_30,
            'shap_values': shap_values,
            'X_sample': X_sample,
            'feature_cols': feature_cols
        }

    except Exception as e:
        print(f"   ❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# EXECUTE ANALYSIS FOR ALL STOCKS
# ============================================================================

print("🚀 Starting Feature Importance Analysis...\n")

results = {}
for stock in STOCKS:
    result = analyze_stock_features(stock)
    if result:
        results[stock] = result

# ============================================================================
# CROSS-STOCK ANALYSIS
# ============================================================================

if len(results) > 0:
    print("\n" + "="*80)
    print("📊 CROSS-STOCK FEATURE ANALYSIS")
    print("="*80)

    # Collect top 30 features from each stock
    all_top_features = {}
    for stock, data in results.items():
        clean_name = stock.replace('.NS', '')
        top_features = data['top_30_features']['feature'].tolist()
        all_top_features[clean_name] = top_features

    # Count frequency of features appearing in top 30
    from collections import Counter
    all_features_flat = [f for features in all_top_features.values() for f in features]
    feature_frequency = Counter(all_features_flat)

    # Universal features (appear in top 30 for multiple stocks)
    universal_features = pd.DataFrame([
        {'feature': f, 'count': c}
        for f, c in feature_frequency.items()
        if c >= 3  # Appears in at least 3 stocks
    ]).sort_values('count', ascending=False)

    print(f"\n🌟 UNIVERSAL FEATURES (appear in top 30 for ≥3 stocks):")
    print(universal_features.head(20).to_string(index=False))

    # Save universal features
    universal_features.to_csv(RESULTS_DAY3 / 'universal_features.csv', index=False)

    # Create heatmap of top features across stocks
    print("\n📈 Creating feature heatmap...")

    # Get top 50 universal features
    top_universal = universal_features.head(50)['feature'].tolist()

    # Create matrix: stocks × features
    heatmap_data = []
    stock_names = []

    for stock, data in results.items():
        clean_name = stock.replace('.NS', '')
        stock_names.append(clean_name)

        # Get importance for top universal features
        feature_imp = data['top_30_features'].set_index('feature')['importance']
        row = [feature_imp.get(f, 0) for f in top_universal]
        heatmap_data.append(row)

    heatmap_df = pd.DataFrame(heatmap_data, columns=top_universal, index=stock_names)

    # Plot heatmap
    plt.figure(figsize=(20, 10))
    sns.heatmap(heatmap_df.T, cmap='YlOrRd', annot=False, cbar_kws={'label': 'Importance'})
    plt.xlabel('Stock', fontsize=14)
    plt.ylabel('Feature', fontsize=14)
    plt.title('Feature Importance Heatmap Across Stocks', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DAY3 / 'feature_heatmap.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("   ✅ Heatmap saved!")

    # Feature categories analysis
    print("\n📊 Analyzing feature categories...")

    categories = {
        'RSI': 'rsi_',
        'MACD': 'macd_',
        'Bollinger': 'bb_',
        'Moving Avg': ['sma_', 'ema_'],
        'Volatility': 'volatility_',
        'Volume': 'volume_',
        'Momentum': ['momentum_', 'roc_'],
        'ATR': 'atr_',
        'Stochastic': 'stoch_'
    }

    category_importance = {}
    for cat_name, patterns in categories.items():
        if isinstance(patterns, str):
            patterns = [patterns]

        cat_features = [f for f in top_universal if any(p in f for p in patterns)]
        category_importance[cat_name] = len(cat_features)

    # Plot category distribution
    plt.figure(figsize=(12, 6))
    cats = list(category_importance.keys())
    counts = list(category_importance.values())
    plt.bar(cats, counts, color='skyblue', edgecolor='navy', alpha=0.7)
    plt.xlabel('Feature Category', fontsize=12)
    plt.ylabel('Count in Top 50 Universal Features', fontsize=12)
    plt.title('Distribution of Feature Categories', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(RESULTS_DAY3 / 'category_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()

    print("   ✅ Category analysis saved!")

    # Save summary
    summary = {
        'total_stocks_analyzed': len(results),
        'total_universal_features': len(universal_features),
        'category_distribution': category_importance,
        'top_10_universal': universal_features.head(10).to_dict('records')
    }

    with open(RESULTS_DAY3 / 'analysis_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    # Final summary
    print("\n" + "="*80)
    print("🎉 DAY 3 COMPLETE!")
    print("="*80)
    print(f"✅ Stocks analyzed: {len(results)}/9")
    print(f"✅ Universal features found: {len(universal_features)}")
    print(f"✅ Top category: {max(category_importance, key=category_importance.get)}")
    print(f"✅ Files created: {len(list(RESULTS_DAY3.glob('*')))}")
    print("="*80)
    print("\n📂 Results saved in: /MarketLab_BEAST/results_day3/")
    print("\n📊 Key files created:")
    print("   • Individual stock SHAP plots (18 PNG files)")
    print("   • Feature importance CSVs (9 files)")
    print("   • Feature heatmap across stocks")
    print("   • Category distribution plot")
    print("   • Universal features list")
    print("   • Analysis summary JSON")
    print("\n🎯 KEY FINDINGS:")
    print(f"   • Top 3 universal features:")
    for idx, row in universal_features.head(3).iterrows():
        print(f"      {idx+1}. {row['feature']} (appears in {row['count']} stocks)")

    print("\n✅ READY FOR DAY 4 - BACKTESTING!")

else:
    print("\n❌ No stocks completed successfully.")

🔥 MARKETLAB DAY 3 - SHAP EXPLAINABILITY ANALYSIS
📊 Analyzing feature importance across 9 stocks
🎯 Understanding which features drive predictions

🚀 Starting Feature Importance Analysis...


Stock: SUNPHARMA.NS
   📥 Loading data...
      ✅ 5195 rows loaded
   🔧 Generating features...
      ✅ Train: 4156 | Test: 1039
   🌲 Training Random Forest...
      ✅ RF R² = 0.1823
   🔍 Computing SHAP values...
      ✅ SHAP computed for 100 samples

   🏆 TOP 10 MOST IMPORTANT FEATURES:
      ema_3                          445.758587
      sma_3                          19.921084
      vwap_5                         18.511945
      close_lag_1                    4.990835
      distance_low_100               2.625994
      ema_5                          2.035699
      kijun_sen                      1.755736
      vwap_20                        1.482758
      ema_20                         1.435367
      ema_200                        1.429476
   ✅ DONE! Plots saved.

Stock: ITC.NS
   📥 Loading data...

In [3]:
# ============================================================================
# MARKETLAB DAY 3 - FUNCTION DEFINITIONS ONLY (FOR ASIANPAINT)
# ============================================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor

print("📚 Loading function definitions...")

# Paths
BASE_PATH = Path('/content/drive/MyDrive/MarketLab_BEAST')
DATA_PATH = BASE_PATH / 'stock_data'
RESULTS_DAY3 = BASE_PATH / 'results_day3'
RESULTS_DAY3.mkdir(exist_ok=True, parents=True)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_csv(ticker):
    """Load stock data"""
    path = DATA_PATH / f"{ticker}.csv"

    with open(path, 'r') as f:
        first_line = f.readline()

    if 'Price","Open' in first_line:
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
        df = df.set_index('Date')
        df = df.rename(columns={'Price': 'Close', 'Vol.': 'Volume'})
        if df['Volume'].dtype == 'object':
            df['Volume'] = df['Volume'].str.replace('M', '').str.replace('K', '').str.replace(',', '')
            df['Volume'] = df['Volume'].apply(lambda x: float(x)*1000000 if 'M' in str(x) else (float(x)*1000 if 'K' in str(x) else float(x) if x else 0))
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    else:
        df = pd.read_csv(path, skiprows=[1])
        df = df.iloc[1:]
        df['Date'] = pd.to_datetime(df['Price'])
        df = df.set_index('Date')
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df.dropna().sort_index()

def generate_325_features(df):
    """Generate all 325 features"""

    # Price features
    df['hl_ratio'] = df['High'] / df['Low']
    df['co_ratio'] = df['Close'] / df['Open']
    df['oc_diff'] = df['Open'] - df['Close']
    df['hl_diff'] = df['High'] - df['Low']
    df['body_size'] = abs(df['Close'] - df['Open'])
    df['upper_shadow'] = df['High'] - df[['Open','Close']].max(axis=1)
    df['lower_shadow'] = df[['Open','Close']].min(axis=1) - df['Low']
    df['body_position'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])

    # Returns
    for p in [1,2,3,5,7,10,14,21,30,60,90,120,180,252]:
        df[f'ret_{p}'] = df['Close'].pct_change(p)
        df[f'log_ret_{p}'] = np.log(df['Close'] / df['Close'].shift(p))

    # Moving averages
    for p in [3,5,7,10,14,20,30,50,100,150,200]:
        df[f'sma_{p}'] = df['Close'].rolling(p).mean()
        df[f'ema_{p}'] = df['Close'].ewm(span=p).mean()
        df[f'close_sma_ratio_{p}'] = df['Close'] / df[f'sma_{p}']
        df[f'close_ema_ratio_{p}'] = df['Close'] / df[f'ema_{p}']

    # Volatility
    for p in [5,10,20,30,60,90]:
        df[f'volatility_{p}'] = df['Close'].pct_change().rolling(p).std()
        df[f'volatility_{p}_annual'] = df[f'volatility_{p}'] * np.sqrt(252)
        df[f'parkinson_{p}'] = np.sqrt((1/(4*np.log(2))) * ((np.log(df['High']/df['Low']))**2).rolling(p).mean())
        df[f'garman_klass_{p}'] = np.sqrt(0.5 * ((np.log(df['High']/df['Low']))**2).rolling(p).mean() - (2*np.log(2)-1) * ((np.log(df['Close']/df['Open']))**2).rolling(p).mean())

    # RSI
    for p in [7,9,14,21,28]:
        delta = df['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(p).mean()
        loss = -delta.where(delta < 0, 0).rolling(p).mean()
        rs = gain / loss
        df[f'rsi_{p}'] = 100 - (100 / (1 + rs))
        df[f'rsi_{p}_slope'] = df[f'rsi_{p}'].diff(5)
        df[f'rsi_{p}_ema'] = df[f'rsi_{p}'].ewm(span=5).mean()

    # MACD
    for fast, slow, signal in [(12,26,9), (5,35,5), (19,39,9)]:
        ema_fast = df['Close'].ewm(span=fast).mean()
        ema_slow = df['Close'].ewm(span=slow).mean()
        df[f'macd_{fast}_{slow}'] = ema_fast - ema_slow
        df[f'macd_signal_{fast}_{slow}_{signal}'] = df[f'macd_{fast}_{slow}'].ewm(span=signal).mean()
        df[f'macd_hist_{fast}_{slow}_{signal}'] = df[f'macd_{fast}_{slow}'] - df[f'macd_signal_{fast}_{slow}_{signal}']
        df[f'macd_slope_{fast}_{slow}'] = df[f'macd_{fast}_{slow}'].diff(3)

    # Bollinger Bands
    for p in [10,20,30,50]:
        for std_mult in [1.5, 2.0, 2.5]:
            sma = df['Close'].rolling(p).mean()
            std = df['Close'].rolling(p).std()
            df[f'bb_upper_{p}_{std_mult}'] = sma + (std_mult * std)
            df[f'bb_lower_{p}_{std_mult}'] = sma - (std_mult * std)
            df[f'bb_width_{p}_{std_mult}'] = (df[f'bb_upper_{p}_{std_mult}'] - df[f'bb_lower_{p}_{std_mult}']) / sma
            df[f'bb_position_{p}_{std_mult}'] = (df['Close'] - df[f'bb_lower_{p}_{std_mult}']) / (df[f'bb_upper_{p}_{std_mult}'] - df[f'bb_lower_{p}_{std_mult}'])

    # Volume
    for p in [5,10,20,30,50]:
        df[f'volume_sma_{p}'] = df['Volume'].rolling(p).mean()
        df[f'volume_ratio_{p}'] = df['Volume'] / df[f'volume_sma_{p}']
        df[f'volume_std_{p}'] = df['Volume'].rolling(p).std()
        df[f'volume_cv_{p}'] = df[f'volume_std_{p}'] / df[f'volume_sma_{p}']
        df[f'obv_{p}'] = (np.sign(df['Close'].diff()) * df['Volume']).rolling(p).sum()
        df[f'vwap_{p}'] = (df['Close'] * df['Volume']).rolling(p).sum() / df['Volume'].rolling(p).sum()

    # Momentum
    for p in [5,10,20,30,50]:
        df[f'momentum_{p}'] = df['Close'] - df['Close'].shift(p)
        df[f'roc_{p}'] = df['Close'].pct_change(p) * 100
        df[f'trix_{p}'] = df['Close'].ewm(span=p).mean().ewm(span=p).mean().ewm(span=p).mean().pct_change()

    # ATR
    for p in [7,14,21,30]:
        high_low = df['High'] - df['Low']
        high_close = np.abs(df['High'] - df['Close'].shift())
        low_close = np.abs(df['Low'] - df['Close'].shift())
        ranges = pd.concat([high_low, high_close, low_close], axis=1)
        true_range = ranges.max(axis=1)
        df[f'atr_{p}'] = true_range.rolling(p).mean()
        df[f'atr_ratio_{p}'] = df[f'atr_{p}'] / df['Close']
        df[f'natr_{p}'] = (df[f'atr_{p}'] / df['Close']) * 100

    # Statistical
    for p in [10,20,30,50]:
        df[f'std_{p}'] = df['Close'].rolling(p).std()
        df[f'skew_{p}'] = df['Close'].rolling(p).skew()
        df[f'kurt_{p}'] = df['Close'].rolling(p).kurt()
        df[f'median_{p}'] = df['Close'].rolling(p).median()
        df[f'quantile_25_{p}'] = df['Close'].rolling(p).quantile(0.25)
        df[f'quantile_75_{p}'] = df['Close'].rolling(p).quantile(0.75)
        df[f'iqr_{p}'] = df[f'quantile_75_{p}'] - df[f'quantile_25_{p}']

    # Lags
    for lag in [1,2,3,5,7,10,14,21,30]:
        df[f'close_lag_{lag}'] = df['Close'].shift(lag)
        df[f'volume_lag_{lag}'] = df['Volume'].shift(lag)
        df[f'high_lag_{lag}'] = df['High'].shift(lag)
        df[f'low_lag_{lag}'] = df['Low'].shift(lag)

    # Support/Resistance
    for p in [10,20,50,100]:
        df[f'high_max_{p}'] = df['High'].rolling(p).max()
        df[f'low_min_{p}'] = df['Low'].rolling(p).min()
        df[f'distance_high_{p}'] = (df['Close'] - df[f'high_max_{p}']) / df['Close']
        df[f'distance_low_{p}'] = (df['Close'] - df[f'low_min_{p}']) / df['Close']

    # Stochastic
    for p in [14,21]:
        low_min = df['Low'].rolling(p).min()
        high_max = df['High'].rolling(p).max()
        df[f'stoch_k_{p}'] = 100 * (df['Close'] - low_min) / (high_max - low_min)
        df[f'stoch_d_{p}'] = df[f'stoch_k_{p}'].rolling(3).mean()

    # Ichimoku
    high_9 = df['High'].rolling(9).max()
    low_9 = df['Low'].rolling(9).min()
    df['tenkan_sen'] = (high_9 + low_9) / 2
    high_26 = df['High'].rolling(26).max()
    low_26 = df['Low'].rolling(26).min()
    df['kijun_sen'] = (high_26 + low_26) / 2
    df['senkou_span_a'] = ((df['tenkan_sen'] + df['kijun_sen']) / 2).shift(26)
    high_52 = df['High'].rolling(52).max()
    low_52 = df['Low'].rolling(52).min()
    df['senkou_span_b'] = ((high_52 + low_52) / 2).shift(26)
    df['chikou_span'] = df['Close'].shift(-26)

    # Clean infinity/NaN
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(method='ffill').fillna(method='bfill').fillna(0)

    return df

def analyze_stock_features(ticker):
    """Analyze feature importance for one stock using Random Forest"""

    print(f"\n{'='*80}")
    print(f"Stock: {ticker}")
    print(f"{'='*80}")

    try:
        # Load data
        print("   📥 Loading data...")
        df = load_csv(ticker)
        print(f"      ✅ {len(df)} rows loaded")

        # Generate features
        print("   🔧 Generating features...")
        df = generate_325_features(df)
        df = df.dropna()

        # Prepare data
        exclude = ['Open', 'High', 'Low', 'Close', 'Volume']
        feature_cols = [c for c in df.columns if c not in exclude]

        X = df[feature_cols].values
        y = df['Close'].values

        # Train/test split (80/20)
        split_idx = int(0.8 * len(X))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        print(f"      ✅ Train: {len(X_train)} | Test: {len(X_test)}")

        # Train Random Forest
        print("   🌲 Training Random Forest...")
        rf = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        score = rf.score(X_test, y_test)
        print(f"      ✅ RF R² = {score:.4f}")

        # SHAP analysis
        print("   🔍 Computing SHAP values...")
        explainer = shap.TreeExplainer(rf)

        sample_size = min(100, len(X_test))
        sample_idx = np.random.choice(len(X_test), sample_size, replace=False)
        X_sample = X_test[sample_idx]

        shap_values = explainer.shap_values(X_sample)
        print(f"      ✅ SHAP computed for {sample_size} samples")

        # Feature importance
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': mean_abs_shap
        }).sort_values('importance', ascending=False)

        top_30 = feature_importance.head(30)

        print("\n   🏆 TOP 10 MOST IMPORTANT FEATURES:")
        for idx, row in top_30.head(10).iterrows():
            print(f"      {row['feature']:30s} {row['importance']:.6f}")

        # Save results
        clean_ticker = ticker.replace('.NS', '')

        feature_importance.to_csv(
            RESULTS_DAY3 / f"{clean_ticker}_feature_importance.csv",
            index=False
        )

        # SHAP summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(
            shap_values,
            X_sample,
            feature_names=feature_cols,
            max_display=20,
            show=False
        )
        plt.title(f'SHAP Feature Importance - {clean_ticker}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(RESULTS_DAY3 / f"{clean_ticker}_shap_summary.png", dpi=300, bbox_inches='tight')
        plt.close()

        # Feature importance bar plot
        plt.figure(figsize=(12, 10))
        top_20 = top_30.head(20)
        plt.barh(range(len(top_20)), top_20['importance'].values)
        plt.yticks(range(len(top_20)), top_20['feature'].values)
        plt.xlabel('Mean |SHAP Value|', fontsize=12)
        plt.title(f'Top 20 Features - {clean_ticker}', fontsize=16, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(RESULTS_DAY3 / f"{clean_ticker}_top20_features.png", dpi=300, bbox_inches='tight')
        plt.close()

        print(f"   ✅ DONE! Plots saved.")

        return {
            'ticker': ticker,
            'rf_r2': score,
            'top_30_features': top_30,
            'shap_values': shap_values,
            'X_sample': X_sample,
            'feature_cols': feature_cols
        }

    except Exception as e:
        print(f"   ❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

print("✅ All functions loaded!")
print("✅ Ready to analyze ASIANPAINT!")

📚 Loading function definitions...
✅ All functions loaded!
✅ Ready to analyze ASIANPAINT!


In [5]:
# ============================================================================
# ADD ASIANPAINT - SIMPLIFIED (NO DEPENDENCIES)
# ============================================================================

print("\n" + "="*80)
print("📊 ANALYZING ASIANPAINT")
print("="*80)

# Analyze ASIANPAINT
asianpaint_result = analyze_stock_features('ASIANPAINT.NS')

if asianpaint_result:
    print("\n" + "="*80)
    print("✅ ASIANPAINT ANALYSIS COMPLETE!")
    print("="*80)
    print(f"✅ RF R²: {asianpaint_result['rf_r2']:.4f}")
    print(f"✅ Files saved to: /MarketLab_BEAST/results_day3/")
    print("="*80)

    print("\n📊 ASIANPAINT TOP 10 FEATURES:")
    for idx, row in asianpaint_result['top_30_features'].head(10).iterrows():
        print(f"   {row['feature']:30s} {row['importance']:.6f}")

    print("\n" + "="*80)
    print("🎉 DAY 3 NOW COMPLETE WITH 10/10 STOCKS!")
    print("="*80)
    print("✅ All 10 stocks analyzed:")
    print("   1. SUNPHARMA ✅")
    print("   2. ITC ✅")
    print("   3. TCS ✅")
    print("   4. RELIANCE ✅")
    print("   5. INFY ✅")
    print("   6. COALINDIA ✅")
    print("   7. BAJFINANCE ✅")
    print("   8. HDFCBANK ✅")
    print("   9. MARUTI ✅")
    print("   10. ASIANPAINT ✅")
    print("\n📂 All results saved in: /MarketLab_BEAST/results_day3/")
    print("\n✅ READY FOR DAY 3.5 - ENSEMBLE + ATTENTION!")

else:
    print("\n❌ ASIANPAINT failed. Check errors above.")


📊 ANALYZING ASIANPAINT

Stock: ASIANPAINT.NS
   📥 Loading data...
      ✅ 5193 rows loaded
   🔧 Generating features...
      ✅ Train: 4154 | Test: 1039
   🌲 Training Random Forest...
      ✅ RF R² = -11.3396
   🔍 Computing SHAP values...
      ✅ SHAP computed for 100 samples

   🏆 TOP 10 MOST IMPORTANT FEATURES:
      ema_3                          937.297561
      sma_3                          221.738907
      close_lag_1                    59.587155
      low_lag_1                      51.532596
      vwap_5                         24.912679
      ema_5                          21.953301
      distance_low_100               17.951449
      bb_upper_20_2.5                7.865476
      close_sma_ratio_150            5.042640
      high_max_10                    2.616807
   ✅ DONE! Plots saved.

✅ ASIANPAINT ANALYSIS COMPLETE!
✅ RF R²: -11.3396
✅ Files saved to: /MarketLab_BEAST/results_day3/

📊 ASIANPAINT TOP 10 FEATURES:
   ema_3                          937.297561
   sma_3        